# Spire satellite availability: CSDA vs. COSMIC/UCAR

How many Spire satellites have data on any given day, for two products:

* **L1a attitude**
* **L2b POD (precise orbit) SP3**

from two archives:

| Source | Access | Credentials |
|---|---|---|
| NASA CSDA STAC API (`https://csdap.earthdata.nasa.gov/stac/`) | `pystac-client` search | none (the STAC API is public; [`csda-client`](https://github.com/NASA-IMPACT/csda-client) is only needed to log in for orders/downloads) |
| COSMIC/UCAR NRT archive (`https://data.cosmic.ucar.edu/gnss-ro/spire/noaa/nrt/`) | HTML directory listings | none |

Only **metadata** is used (item names, dates, directory/file names). No data files are downloaded.

**How to use.** Run top to bottom.

1. *Discovery* sections (2 and 4) are the scratch analysis. They print what each archive actually contains
   (collection ids, item/asset names, directory layout) and how well the product and satellite parsers match.
2. If the parsers miss files, edit the regexes in the single **Configuration** cell (1) and re-run.
3. The harvest sections (3 and 5) walk the full record and cache results on disk, so reruns are fast.

> The discovery step exists because the exact collection ids and file naming were not verified when this notebook was
> written. Check the *coverage* tables before trusting any plot.

## 1. Configuration

In [ ]:
import html.parser
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import unquote, urljoin, urlparse

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from pystac_client import Client

try:
    from tqdm.auto import tqdm
except ImportError:  # progress bars are optional
    def tqdm(it=None, **kw):
        return it if it is not None else _NoBar()

    class _NoBar:
        def update(self, n=1): pass
        def close(self): pass

# ---- Sources ---------------------------------------------------------------
CSDA_STAC_URL = "https://csdap.earthdata.nasa.gov/stac/"
COSMIC_ROOT = "https://data.cosmic.ucar.edu/gnss-ro/spire/noaa/nrt/"

# None -> auto-detect the Spire collection(s) by name (see section 2). Or e.g. ["spire"].
CSDA_COLLECTIONS = None

# Sub-directories of COSMIC_ROOT to crawl, e.g. ["level1a", "level2b"]. None -> crawl everything
# under the root (slow: it is the whole archive). Set this after the discovery in section 4.
COSMIC_SUBDIRS = None

# Date window. None = whole record available in each archive.
START = None  # e.g. "2023-01-01"
END = None    # e.g. "2024-12-31"

# ---- Parsers (edit after looking at the discovery output) -------------------
# Matched (case-insensitive) against "<item id> <asset key> <file name>" (CSDA)
# or the path relative to COSMIC_ROOT (COSMIC). First matching product wins.
PRODUCT_PATTERNS = {
    "L1a attitude": r"leoAtt|attitude|(?<![a-z0-9])att(?![a-z0-9])",
    "L2b POD SP3": r"sp3|leoOrb|(?<![a-z0-9])pod(?![a-z0-9])|podTc",
}
# Satellite id. Group 1 is the id. Matched against the same text.
SAT_REGEX = r"(?<![A-Za-z0-9])(FM\d{3}|SPIRE[-_ ]?\d{3,4})(?![A-Za-z0-9])"

# ---- Runtime --------------------------------------------------------------
CACHE_DIR = Path("spire_availability_cache")
REFRESH = False       # True ignores the on-disk caches
MAX_WORKERS = 8       # parallel HTTP requests
RECENT_DAYS = 3       # days this recent are never cached (files may still arrive)

CACHE_DIR.mkdir(exist_ok=True)
START_D = date.fromisoformat(START) if START else None
END_D = date.fromisoformat(END) if END else None
PRODUCTS = list(PRODUCT_PATTERNS)
COLORS = {"L1a attitude": "#2a78d6", "L2b POD SP3": "#eb6834", "both": "#52514e"}

### Shared helpers
Product classification, satellite-id extraction, and date parsing are used by both archives, so they are defined once.

In [ ]:
def classify(text):
    """Return the product name for a text blob, or None."""
    for name, pat in PRODUCT_PATTERNS.items():
        if re.search(pat, text, re.I):
            return name
    return None


def find_sat(text):
    """Return a normalised satellite id (e.g. 'FM101') or None."""
    m = re.search(SAT_REGEX, text, re.I)
    if not m:
        return None
    return re.sub(r"[-_ ]", "", m.group(1)).upper()


_DATE_PATTERNS = [
    # year/doy directory pair, or year.doy / year_doy in a name
    (re.compile(r"(?<!\d)((?:19|20)\d{2})[/._-](\d{3})(?!\d)"), "doy"),
    (re.compile(r"(?<!\d)((?:19|20)\d{2})[-/._]?(\d{2})[-/._]?(\d{2})(?!\d)"), "ymd"),
]


def date_from_text(text):
    """Best-effort UTC date from a path or file name; None if not found."""
    for pat, kind in _DATE_PATTERNS:
        for m in pat.finditer(text):
            try:
                if kind == "doy":
                    y, d = int(m.group(1)), int(m.group(2))
                    if 1 <= d <= 366:
                        return date(y, 1, 1) + timedelta(days=d - 1)
                else:
                    return date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            except ValueError:
                continue
    return None


def in_window(d):
    return (START_D is None or d >= START_D) and (END_D is None or d <= END_D)


def cacheable(d):
    return d is not None and d < date.today() - timedelta(days=RECENT_DAYS)


def tidy(rows):
    """rows: iterable of (date, sat, product, n_files) -> aggregated DataFrame."""
    df = pd.DataFrame(rows, columns=["date", "sat", "product", "n_files"])
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"])
    return df.groupby(["date", "sat", "product"], as_index=False)["n_files"].sum()

## 2. CSDA: discovery (scratch analysis)

The CSDA STAC catalog has one collection per vendor. List the collections and pick out the Spire ones.

In [ ]:
stac = Client.open(CSDA_STAC_URL)
try:
    all_collections = list(stac.collection_search().collections_as_dicts())
except Exception as exc:  # server without the collection-search extension
    print("collection_search failed, falling back to get_collections():", exc)
    all_collections = [c.to_dict() for c in stac.get_collections()]

def _is_spire(c):
    blob = f"{c.get('id','')} {c.get('title','')} {c.get('description','')}".lower()
    return "spire" in blob

spire_cols = [c for c in all_collections if _is_spire(c)]
print(f"{len(all_collections)} collections in CSDA, {len(spire_cols)} mention 'spire':")
pd.DataFrame(
    [
        {
            "id": c["id"],
            "title": c.get("title"),
            "temporal_extent": c.get("extent", {}).get("temporal", {}).get("interval"),
        }
        for c in spire_cols
    ]
)

In [ ]:
if CSDA_COLLECTIONS is None:
    CSDA_COLLECTIONS = [c["id"] for c in spire_cols]
if not CSDA_COLLECTIONS:
    raise RuntimeError(
        "No Spire collection found. Inspect `all_collections` above and set CSDA_COLLECTIONS by hand."
    )
CSDA_EXTENT = {
    c["id"]: c.get("extent", {}).get("temporal", {}).get("interval", [[None, None]])[0]
    for c in all_collections
    if c["id"] in CSDA_COLLECTIONS
}
print("Using collections:", CSDA_COLLECTIONS)
print("Temporal extents:", CSDA_EXTENT)

Look at a handful of real items: their ids, properties, and assets. This is what the product/satellite parsers have to work with.

In [ ]:
def peek_items(collection, n=5, **search_kw):
    search = stac.search(collections=[collection], max_items=n, **search_kw)
    for item in search.items():
        print("item id    :", item.id)
        print("datetime   :", item.datetime, "| start/end:", item.properties.get("start_datetime"), item.properties.get("end_datetime"))
        print("properties :", {k: v for k, v in item.properties.items() if not isinstance(v, (dict, list))})
        for key, a in item.assets.items():
            print(f"  asset {key!r}: {a.href}  ({a.media_type})")
        print()

for col in CSDA_COLLECTIONS:
    print("=" * 20, col)
    peek_items(col, n=3)

Parser check on a few hundred items from each collection: how many get a product and a satellite id? **Anything unmatched
must be fixed in the Configuration cell before the harvest.**

In [ ]:
def item_records(item):
    """Yield (date, sat, product, name) for each classified asset (or the item itself)."""
    props = item.properties
    dt_ = item.datetime or props.get("start_datetime")
    if isinstance(dt_, str):
        dt_ = datetime.fromisoformat(dt_.replace("Z", "+00:00"))
    day = dt_.astimezone(timezone.utc).date() if dt_ else None
    plat = props.get("platform")
    plat = " ".join(plat) if isinstance(plat, (list, tuple)) else (plat or "")
    cands = list(item.assets.items()) or [(None, None)]
    seen = set()
    for key, asset in cands:
        href = unquote(urlparse(asset.href).path.rsplit("/", 1)[-1]) if asset is not None else ""
        text = f"{item.id} {key or ''} {href}"
        product = classify(text) or classify(plat)
        sat = find_sat(text) or find_sat(plat)
        d = day or date_from_text(text)
        rec = (d, sat, product, href or item.id)
        if rec not in seen:
            seen.add(rec)
            yield rec


def csda_sample_report(collection, n=300):
    rows = []
    for item in stac.search(collections=[collection], max_items=n).items():
        rows += [dict(zip(["date", "sat", "product", "name"], r)) for r in item_records(item)]
    df = pd.DataFrame(rows)
    print(f"{collection}: {len(df)} records from sample")
    if df.empty:
        return df
    df["product"] = df["product"].fillna("UNMATCHED")
    df["sat"] = df["sat"].fillna("UNMATCHED")
    display(pd.crosstab(df["product"], df["sat"]))
    bad = df[(df["product"] == "UNMATCHED") | (df["sat"] == "UNMATCHED") | df["date"].isna()]
    if len(bad):
        print(f"{len(bad)} records with a missing product / satellite / date, e.g.:")
        display(bad.head(10))
    return df

csda_samples = {col: csda_sample_report(col) for col in CSDA_COLLECTIONS}

## 3. CSDA: harvest the whole record

Search one calendar month at a time, per collection. Each month is reduced to *(date, satellite, product, number of files)* and
cached to disk, so the first run is slow and later ones only fetch new or recent months.

In [ ]:
def month_chunks(first, last):
    d = first.replace(day=1)
    while d <= last:
        nxt = (d.replace(day=28) + timedelta(days=4)).replace(day=1)
        yield d, nxt - timedelta(days=1)
        d = nxt


def _parse_extent_date(s, default):
    if not s:
        return default
    return datetime.fromisoformat(s.replace("Z", "+00:00")).date()


def harvest_csda_chunk(collection, first, last):
    """One month of one collection -> (rows, diagnostics)."""
    cache = CACHE_DIR / f"csda_{collection}_{first:%Y%m}.csv"
    if cache.exists() and not REFRESH:
        return pd.read_csv(cache).values.tolist(), {"unmatched": 0, "items": None}
    rng = f"{first:%Y-%m-%d}T00:00:00Z/{last:%Y-%m-%d}T23:59:59Z"
    rows, unmatched, n_items = [], 0, 0
    for attempt in range(4):
        try:
            rows, unmatched, n_items = [], 0, 0
            for item in stac.search(collections=[collection], datetime=rng, limit=1000).items():
                n_items += 1
                for d, sat, product, _ in item_records(item):
                    if sat is None or product is None or d is None:
                        unmatched += 1
                    else:
                        rows.append((d.isoformat(), sat, product, 1))
            break
        except Exception as exc:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    agg = tidy(rows)
    out = agg.assign(date=agg["date"].dt.strftime("%Y-%m-%d")).values.tolist() if len(agg) else []
    if cacheable(last):
        pd.DataFrame(out, columns=["date", "sat", "product", "n_files"]).to_csv(cache, index=False)
    return out, {"unmatched": unmatched, "items": n_items}


def harvest_csda():
    jobs = []
    for col in CSDA_COLLECTIONS:
        lo, hi = CSDA_EXTENT.get(col, (None, None))
        first = max(filter(None, [_parse_extent_date(lo, date(2018, 1, 1)), START_D]))
        last = min(filter(None, [_parse_extent_date(hi, date.today()), END_D, date.today()]))
        jobs += [(col, a, b) for a, b in month_chunks(first, last)]
    rows, unmatched = [], 0
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = [ex.submit(harvest_csda_chunk, *j) for j in jobs]
        for f in tqdm(futs, desc="CSDA months"):
            r, diag = f.result()
            rows += r
            unmatched += diag["unmatched"]
    print(f"records without product/satellite/date (dropped): {unmatched}")
    return tidy(rows)

csda = harvest_csda()
csda.groupby("product")["n_files"].agg(["sum", "count"]).rename(columns={"sum": "files", "count": "date-sat rows"})

## 4. COSMIC/UCAR: discovery (scratch analysis)

Walk the top of the archive tree and print the first few entries at each level so the layout (product directories,
year / day-of-year folders, file naming) is visible.

In [ ]:
_session = requests.Session()
_session.headers["User-Agent"] = "contigo-edr-availability-notebook"
_adapter = requests.adapters.HTTPAdapter(
    max_retries=requests.adapters.Retry(total=4, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504]),
    pool_maxsize=MAX_WORKERS * 2,
)
_session.mount("https://", _adapter)
_session.mount("http://", _adapter)


class _Links(html.parser.HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for k, v in attrs:
                if k == "href" and v:
                    self.hrefs.append(v)


def list_dir(url):
    """Return (subdir_urls, file_urls) for an Apache/nginx style index page."""
    r = _session.get(url, timeout=60)
    r.raise_for_status()
    p = _Links()
    p.feed(r.text)
    subdirs, files = [], []
    base = urlparse(url)
    for href in p.hrefs:
        if href.startswith(("?", "#", "mailto:")) or href in ("../", "./", "/"):
            continue
        full = urljoin(url, href)
        u = urlparse(full)
        if u.netloc != base.netloc or not u.path.startswith(base.path) or u.path == base.path:
            continue  # parent links, external links
        (subdirs if href.endswith("/") else files).append(full)
    return sorted(set(subdirs)), sorted(set(files))


def describe_tree(url, depth=4, width=6, indent=0):
    subdirs, files = list_dir(url)
    pad = "  " * indent
    print(f"{pad}{url.removeprefix(COSMIC_ROOT) or '/'}  [{len(subdirs)} dirs, {len(files)} files]")
    for f in files[:width]:
        print(f"{pad}  - {unquote(f.rsplit('/', 1)[-1])}")
    if depth > 0 and subdirs:
        # descend into the last (most recent) entry so file names are visible
        describe_tree(subdirs[-1], depth - 1, width, indent + 1)

root_dirs, root_files = list_dir(COSMIC_ROOT)
print("Top level of", COSMIC_ROOT)
for u in root_dirs:
    print("  dir ", u.removeprefix(COSMIC_ROOT))
for u in root_files:
    print("  file", u.removeprefix(COSMIC_ROOT))

In [ ]:
for u in root_dirs:
    print("-" * 70)
    try:
        describe_tree(u, depth=3)
    except requests.RequestException as exc:
        print(u, "->", exc)

From the tree above, decide which top-level folders hold the **L1a attitude** and **L2b POD SP3** files and set
`COSMIC_SUBDIRS` in the Configuration cell (for example `["level1a", "level2b"]`). The cell below lists which top-level
folders the product patterns would pick up automatically and tests the parsers on a recent day.

In [ ]:
auto = [u for u in root_dirs if classify(u.removeprefix(COSMIC_ROOT))]
print("Top-level folders whose name matches a product pattern:", [u.removeprefix(COSMIC_ROOT) for u in auto])

def sample_leaf(url, max_depth=6):
    """Walk down the last entry at each level until a directory with files is found."""
    for _ in range(max_depth):
        subdirs, files = list_dir(url)
        if files:
            return url, files
        if not subdirs:
            return url, []
        url = subdirs[-1]
    return url, []

rows = []
for u in (COSMIC_SUBDIRS and [urljoin(COSMIC_ROOT, s.strip("/") + "/") for s in COSMIC_SUBDIRS]) or root_dirs:
    leaf, files = sample_leaf(u)
    for f in files[:400]:
        rel = unquote(f.removeprefix(COSMIC_ROOT))
        rows.append({"path": rel, "product": classify(rel), "sat": find_sat(rel), "date": date_from_text(rel)})
sample = pd.DataFrame(rows)
if len(sample):
    display(sample.fillna("UNMATCHED").sample(min(15, len(sample)), random_state=0))
    print("Unmatched: product", sample["product"].isna().sum(), "| sat", sample["sat"].isna().sum(),
          "| date", sample["date"].isna().sum(), "of", len(sample))

## 5. COSMIC/UCAR: harvest the whole record

Breadth-first crawl, one level at a time and in parallel. Directories that contain files are reduced to
*(date, satellite, product, number of files)*; the results are cached per directory and recent days
are always re-listed.

In [ ]:
import json

COSMIC_CACHE = CACHE_DIR / "cosmic_dirs.json"
_dir_cache = {} if REFRESH or not COSMIC_CACHE.exists() else json.loads(COSMIC_CACHE.read_text())


def _skip_by_window(rel):
    """Prune year or year/doy paths that are completely outside START/END."""
    parts = [p for p in rel.strip("/").split("/") if p]
    for p in parts:
        if re.fullmatch(r"(19|20)\d{2}", p):
            y = int(p)
            if (START_D and y < START_D.year) or (END_D and y > END_D.year):
                return True
    d = date_from_text(rel.strip("/") + "/")
    return d is not None and not in_window(d)


def scan_dir(url):
    """-> (subdirs, rows, stats). Rows are (date, sat, product, n_files) for files in this dir."""
    rel = unquote(url.removeprefix(COSMIC_ROOT))
    if url in _dir_cache:
        c = _dir_cache[url]
        return c["subdirs"], [tuple(r) for r in c["rows"]], c["stats"]
    subdirs, files = list_dir(url)
    rows, stats = [], {"files": len(files), "unmatched": 0}
    for f in files:
        name = unquote(f.removeprefix(COSMIC_ROOT))
        product, sat = classify(name), find_sat(name)
        d = date_from_text(rel) or date_from_text(name.rsplit("/", 1)[-1])
        if product is None or sat is None or d is None:
            stats["unmatched"] += 1
            continue
        if in_window(d):
            rows.append((d.isoformat(), sat, product, 1))
    day = date_from_text(rel)
    if not subdirs and cacheable(day):
        _dir_cache[url] = {"subdirs": [], "rows": rows, "stats": stats}
    return subdirs, rows, stats


def harvest_cosmic():
    starts = (
        [urljoin(COSMIC_ROOT, s.strip("/") + "/") for s in COSMIC_SUBDIRS]
        if COSMIC_SUBDIRS else [COSMIC_ROOT]
    )
    frontier, all_rows, n_files, n_unmatched, level = starts, [], 0, 0, 0
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        while frontier and level < 10:
            bar = tqdm(total=len(frontier), desc=f"COSMIC level {level}")
            nxt = []
            for subdirs, rows, stats in ex.map(scan_dir, frontier):
                all_rows += rows
                n_files += stats["files"]
                n_unmatched += stats["unmatched"]
                nxt += [s for s in subdirs if not _skip_by_window(unquote(s.removeprefix(COSMIC_ROOT)))]
                bar.update(1)
            bar.close()
            frontier, level = nxt, level + 1
    COSMIC_CACHE.write_text(json.dumps(_dir_cache))
    print(f"files seen: {n_files:,} | not classified as attitude/POD or missing satellite/date: {n_unmatched:,}")
    return tidy(all_rows)

cosmic = harvest_cosmic()
cosmic.groupby("product")["n_files"].agg(["sum", "count"]).rename(columns={"sum": "files", "count": "date-sat rows"})

## 6. Satellites per day

For each source, the number of **distinct satellites with at least one file that day**, per product. The grey line is the number
of satellites that have **both** products on that day, which is what a downstream EDR calculation needs.

In [ ]:
def daily_counts(df):
    """Per-day number of satellites for each product, and for both products together."""
    if df.empty:
        return pd.DataFrame(columns=PRODUCTS + ["both"])
    p = df.pivot_table(index=["date", "sat"], columns="product", values="n_files", aggfunc="sum").reindex(columns=PRODUCTS)
    present = p.notna()
    out = present.groupby(level="date").sum().astype(int)
    out["both"] = present.all(axis=1).groupby(level="date").sum().astype(int)
    full = pd.date_range(out.index.min(), out.index.max(), freq="D")
    return out.reindex(full, fill_value=0)

sources = {"CSDA (STAC)": csda, "COSMIC/UCAR NRT": cosmic}
counts = {name: daily_counts(df) for name, df in sources.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True, constrained_layout=True)
for ax, (name, c) in zip(axes, counts.items()):
    for key in PRODUCTS:
        if key in c:
            ax.plot(c.index, c[key], color=COLORS[key], lw=1.6, label=key)
    if "both" in c:
        ax.plot(c.index, c["both"], color=COLORS["both"], lw=1.4, ls="--", label="both products")
    ax.set_title(name, loc="left", fontsize=12)
    ax.set_ylabel("Satellites with data")
    ax.grid(True, color="0.88", lw=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
    ax.legend(frameon=False, loc="upper left")
    ax.set_ylim(bottom=0)
fig.suptitle("Spire satellites available per day", x=0.01, ha="left", fontsize=14)
plt.show()

### Which satellites, and when
One row per satellite, one column per day, for each product, in both archives (filled = at least one file that day).

In [ ]:
def presence_matrix(df, product):
    d = df[df["product"] == product]
    if d.empty:
        return pd.DataFrame()
    m = d.pivot_table(index="sat", columns="date", values="n_files", aggfunc="sum").notna()
    full = pd.date_range(m.columns.min(), m.columns.max(), freq="D")
    return m.reindex(columns=full, fill_value=False).sort_index()

fig, axes = plt.subplots(len(PRODUCTS), 2, figsize=(14, 3.2 * len(PRODUCTS)), constrained_layout=True)
axes = np.atleast_2d(axes)
for j, (name, df) in enumerate(sources.items()):
    for i, prod in enumerate(PRODUCTS):
        ax = axes[i, j]
        m = presence_matrix(df, prod)
        ax.set_title(f"{name}: {prod}", loc="left", fontsize=11)
        if m.empty:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        x = mdates.date2num(m.columns.to_pydatetime())
        ax.pcolormesh(
            np.append(x, x[-1] + 1), np.arange(len(m) + 1), m.values.astype(float),
            cmap=plt.matplotlib.colors.ListedColormap(["#f1f0ee", COLORS[prod]]), vmin=0, vmax=1,
        )
        ax.set_yticks(np.arange(len(m)) + 0.5)
        ax.set_yticklabels(m.index, fontsize=7)
        ax.xaxis_date()
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Summary table

In [ ]:
def summarise(name, df, c):
    rows = []
    for prod in PRODUCTS + ["both"]:
        if prod not in c or c[prod].sum() == 0:
            rows.append({"source": name, "series": prod})
            continue
        s = c[prod]
        active = s[s > 0]
        rows.append({
            "source": name, "series": prod,
            "first day": active.index.min().date(), "last day": active.index.max().date(),
            "days with data": int((s > 0).sum()),
            "median sats/day": float(active.median()),
            "max sats/day": int(s.max()),
            "distinct sats": df[df["product"] == prod]["sat"].nunique() if prod != "both" else np.nan,
        })
    return rows

pd.DataFrame([r for n, d in sources.items() for r in summarise(n, d, counts[n])]).set_index(["source", "series"])

### Save the daily tables

In [ ]:
for name, df in sources.items():
    tag = re.split(r"\W", name)[0].lower()
    df.to_csv(CACHE_DIR / f"{tag}_date_sat_product.csv", index=False)
    counts[name].rename_axis("date").to_csv(CACHE_DIR / f"{tag}_daily_counts.csv")
print("Saved to", CACHE_DIR.resolve())